In [17]:
# %pip install fastapi
# %pip install fastapi uvicorn
from fastapi import FastAPI
from pathlib import Path
import sys
sys.path.append(str(Path.cwd().parent / "stock_crawler"))
import stock_lib as sl
from datetime import date, datetime
import pandas as pd
import numpy as np

In [6]:
app = FastAPI()

@app.get("/analysis/{ticker}")
def analysis(ticker: str):
    stock = sl.get_stock(ticker)
    return {
        "ticker": ticker,
        "stock": stock
    }

Run via terminal (from the `rest_api` folder):

```powershell
uvicorn app:app --reload
```

In [18]:
result = sl.get_stock("NVDA", date(2026, 6, 1), date(2026, 7, 1))
# propagate error dicts from the library
# convert pandas DataFrame to JSON-serializable list of dicts
if isinstance(result, pd.DataFrame):
    df = result.reset_index()
    # replace NaN (and other missing values) with None so JSON encoder accepts them
    df = df.where(pd.notnull(df), None)
    records = df.to_dict(orient="records")

    def convert(v):
        if isinstance(v, (np.integer,)):
            return int(v)
        if isinstance(v, (np.floating,)):
            return float(v)
        if isinstance(v, (np.bool_,)):
            return bool(v)
        if isinstance(v, (pd.Timestamp, datetime)):
            return v.isoformat()
        return v

    for r in records:
        for k, v in list(r.items()):
            r[k] = convert(v)

In [19]:
records

[{'index': 0,
  'symbol': 'NVDA',
  'date': '2026-06-01T00:00:00',
  'open': 215.47885750139108,
  'close': 224.09881591796875,
  'high': 224.60821672379726,
  'low': 215.44889364463796,
  'volume': 212850700,
  'dividends': 0.0,
  'stock_split': 0.0},
 {'index': 1,
  'symbol': 'NVDA',
  'date': '2026-06-02T00:00:00',
  'open': 226.91551751623658,
  'close': 222.56060791015625,
  'high': 232.00958636747916,
  'low': 221.09231801448135,
  'volume': 193362900,
  'dividends': 0.0,
  'stock_split': 0.0},
 {'index': 2,
  'symbol': 'NVDA',
  'date': '2026-06-03T00:00:00',
  'open': 221.46188713313538,
  'close': 214.5,
  'high': 222.56061267075634,
  'low': 214.2602739078757,
  'volume': 160907000,
  'dividends': 0.0,
  'stock_split': 0.0},
 {'index': 3,
  'symbol': 'NVDA',
  'date': '2026-06-04T00:00:00',
  'open': 213.91000366210938,
  'close': 218.66000366210938,
  'high': 221.60000610351562,
  'low': 210.97000122070312,
  'volume': 169022200,
  'dividends': 0.25,
  'stock_split': 0.0},
 